# YOUR FIRST LAB

### Please read this section. This is valuable to get you prepared, even if it's a long read -- it's important stuff.

## Your first Frontier LLM Project

By the end of this course, you will have built an autonomous Agentic AI solution with 7 agents that collaborate to solve a business problem. All in good time! We will start with something smaller...

Our goal is to code a new kind of Web Browser. Give it a URL, and it will respond with a summary. The Reader's Digest of the internet!!

## Business value of these exercises

A final thought. While we've designed these notebooks to be educational, we've also tried to make them enjoyable. We'll do fun things like have LLMs tell jokes and argue with each other. But fundamentally, our goal is to teach skills you can apply in business. We'll explain business implications as we go, and it's worth keeping this in mind: as you build experience with models and techniques, think of ways you could put this into action at work today.

### If necessary, install Cursor Extensions

1. From the View menu, select Extensions
2. Search for Python
3. Click on "Python" made by "ms-python" and select Install if not already installed
4. Search for Jupyter
5. Click on "Jupyter" made by "ms-toolsai" and select Install if not already installed


### Next Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.

Any problems with this? Head over to the troubleshooting.

### Note: you'll need to set the Kernel with every notebook..

In [12]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to OpenAI (or Ollama)

The next cell is where we load in the environment variables in your `.env` file and connect to OpenAI.  

If you'd like to use free Ollama instead, please see the README section "Free Alternative to Paid APIs", and if you're not sure how to do this, there's a full solution in the solutions folder (day1_with_ollama.ipynb).

## Troubleshooting if you have problems:

If you get a "Name Error" - have you run all cells from the top down? Head over to the Python Foundations guide for a bulletproof way to find and fix all Name Errors.

If that doesn't fix it, head over to the [troubleshooting](../setup/troubleshooting.ipynb) notebook for step by step code to identify the root cause and fix it!

Any concerns about API costs? See notes in the README - costs should be minimal, and you can control it at every point. You can also use Ollama as a free alternative, which we discuss during Day 2.

In [13]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Models have been updated since the video

AI companies retire their older models every few months, so some of the models in this lab are newer than the ones you'll see in the video, and your results will look a bit different. If a model in this course ever stops working, please pick a similar model from that company's latest line-up.

# Let's make a quick call to a Frontier model to get started, as a preview!

In [14]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [15]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-6-luna", reasoning_effort="none", messages=messages)
response.choices[0].message.content

'Hi! Welcome 😊 How can I help you today?'

## OK onwards with our first project

In [16]:
# Let's try out this utility

aw = fetch_website_contents("https://www.techverse51.com/")
print(aw)

TechVerse51 - Innovating the Future of Software

Home
About
Services
Contact
Welcome to TechVerse51
Innovating the
Future of Software
We create cutting-edge solutions that transform businesses and drive technology forward.
Get Started
See Our Work
10+
Projects Completed
15+
Team Members
12+
Months Experience
About Us
Who We Are
TechVerse51 is a pioneering software house driven by a passion for innovation and technological excellence. We craft solutions that help businesses thrive in today's rapidly evolving digital landscape.
Our Values
Quality Code
Clean, efficient solutions that stand the test of time.
Fast Delivery
Efficient processes delivering on time, every time.
Global Reach
Serving clients worldwide with tailored solutions.
Expert Team
Specialists with years of industry experience.
Our Services
Delivering exceptional digital solutions tailored to your business needs
1
Web Development
We build blazing fast, secure, and responsive websites and web apps using cutting-edge tools an

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [17]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [18]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [19]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = openai.chat.completions.create(model="gpt-6-luna", reasoning_effort="none", messages=messages)
response.choices[0].message.content

'2 + 2 = 4.'

## And now let's build useful messages for GPT-6 Luna, using a function

In [20]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [21]:
# Try this out, and then try for a few more websites

messages_for(aw)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': "\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nTechVerse51 - Innovating the Future of Software\n\nHome\nAbout\nServices\nContact\nWelcome to TechVerse51\nInnovating the\nFuture of Software\nWe create cutting-edge solutions that transform businesses and drive technology forward.\nGet Started\nSee Our Work\n10+\nProjects Completed\n15+\nTeam Members\n12+\nMonths Experience\nAbout Us\nWho We Are\nTechVerse51 is a pioneering software house driven by a passion for innovation and technological excellence. We craft solutions that help businesses thrive in today's r

## Time to bring it together - the API for OpenAI is very simple!

In [22]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-6-luna",
        reasoning_effort = "none",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [23]:
summarize("https://www.techverse51.com/")

'TechVerse51 is a software company offering web development, UI/UX design, database management, and system administration. It promises polished tech, fast delivery, and global service—backed by a modest 12 months of experience and a mysteriously truncated service list.'

In [25]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [26]:
display_summary("https://www.techverse51.com/")

TechVerse51 pitches itself as a software house building websites, user interfaces, databases, and IT infrastructure—because apparently the future needs all four. It highlights clean code, fast delivery, global clients, and a team of 15+, with 10+ projects completed and a modest 12+ months of experience. No news or announcements are included.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [27]:
display_summary("https://cnn.com")

CNN’s homepage offers breaking news, live TV, video, and coverage across politics, world affairs, business, health, entertainment, sports, science, climate, and more. The supplied page contains no specific news stories or announcements—mostly menus, account links, and an ad-feedback form bravely demanding feedback about the ads.

In [28]:
display_summary("https://anthropic.com")

Anthropic presents itself as the safety-minded AI company—because building powerful AI apparently comes with a few things to worry about. Its big announcement is **Fable 5.2 and Mythos 5.2**, billed as its strongest models yet for coding, finance, and science. Recent releases also include **Opus 5.5** and **Sonnet 5.5**, with claims of better performance, lower costs, or faster work.

The rest of the site highlights Anthropic’s AI-safety research, policies, and commitments, including a new institute, Claude’s constitution, and a framework for managing catastrophic risks.

# Business applications

In this exercise, you experienced calling the Cloud API of a Frontier Model (a leading model at the frontier of AI) for the first time. We will be using APIs like OpenAI at many stages in the course, in addition to building our own LLMs.

More specifically, we've applied this to Summarization - a classic Gen AI use case to make a summary. This can be applied to any business vertical - summarizing the news, summarizing financial performance, summarizing a resume in a cover letter - the applications are limitless. Consider how you could apply Summarization in your business, and try prototyping a solution.

## Before you continue - now try yourself

Use the cell below to make your own simple commercial example. Stick with the summarization use case for now. Here's an idea: write something that will take the contents of an email, and will suggest an appropriate short subject line for the email. That's the kind of feature that might be built into a commercial email tool.

In [ ]:
# Step 1: Create your prompts

system_prompt = "something here"
user_prompt = """
    Lots of text
    Can be pasted here
"""

# Step 2: Make the messages list

messages = [] # fill this in

# Step 3: Call OpenAI
# response =

# Step 4: print the result
# print(

## An extra exercise for those who enjoy web scraping

You may notice that if you try `display_summary("https://openai.com")` - it doesn't work! That's because OpenAI has a fancy website that uses Javascript. There are many ways around this that some of you might be familiar with. For example, Selenium is a hugely popular framework that runs a browser behind the scenes, renders the page, and allows you to query it. If you have experience with Selenium, Playwright or a similar browser automation framework, then feel free to implement an alternative version of the scraping functions in `scraper.py` (or in a separate module) using one of these tools. In the community-contributions folder, you'll find examples contributed by other students, including a Selenium implementation (thank you!)

# Sharing your code

We'd love it if you share your code afterwards so we can share it with others! You'll notice that some students have already made changes (including a Selenium implementation) which you will find in the community-contributions folder. If you'd like add your changes to that folder, submit a Pull Request with your new versions in that folder and we'll merge your changes.

If you're not an expert with git (and many of us are not!) then complete instructions are available in the guides folder.

Please check before submitting:  
1. Your PR only contains changes in community-contributions (unless we've discussed it)  
2. All notebook outputs are clear  
3. Less than 2,000 lines of code in total, and not too many files  
4. Don't include unnecessary test files, or overly wordy README or .env.example or emojis or other LLM artifacts!

Thanks so much!